# SIMPLIcity CNN Image Classification Experiment
### Course: Neuro Computing (PENS) - Case Study: Image Classification with SIMPLIcity Dataset

This notebook provides a complete, reproducible workflow to train and evaluate CNN models on Google Colab or Kaggle with GPU acceleration.

**Models Evaluated:**
1. `baseline_custom`: 2-layer CNN baseline as defined in Slide 6 of the assignment.
2. `custom_scratch_regularized`: Custom CNN from scratch with EDA mitigations (ColorJitter, RandomErasing, CropScale, Label Smoothing, and Macro F1 monitoring).
3. `resnet18_transfer`: Transfer learning benchmark using ImageNet pretrained backbone.

## 1. Verify GPU Acceleration
Make sure runtime is set to **GPU** (`Runtime -> Change runtime type -> T4 GPU` in Colab).

In [ ]:
!nvidia-smi
import torch

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name:     {torch.cuda.get_device_name(0)}")

## 2. Setup Environment & Repository
Clone the official repository (branch `main`) if running in a fresh cloud instance (Google Colab / Kaggle), then install the project in editable mode.

In [ ]:
import os

# 1. Ensure working directory is the project root (auto-clone if in fresh Colab/Kaggle)
if os.path.exists("src/simplicity_cnn"):
    print("Already inside cnn-SIMPLIcity project root.")
elif os.path.exists("../src/simplicity_cnn"):
    print("Currently in notebooks/ subdirectory. Moving to project root...")
    %cd ..
elif os.path.exists("cnn-SIMPLIcity/src/simplicity_cnn"):
    print("cnn-SIMPLIcity folder found. Entering project directory...")
    %cd cnn-SIMPLIcity
else:
    print("Repository structure not found in VM. Cloning branch 'main' from GitHub...")
    !git clone -b main https://github.com/Arfiadi/cnn-SIMPLIcity.git
    %cd cnn-SIMPLIcity

# 2. Install project in editable mode
%pip install -e .
%pip install -r requirements.txt

## 3. Dataset Setup
Automatically detects dataset in Kaggle input (`/kaggle/input/...`) or extracts `simplicity_dataset.zip` if running on Colab/Local.

In [ ]:
import os
import zipfile
from pathlib import Path

from simplicity_cnn.utils.env import get_default_data_dir

# 1. Attempt auto-detection across Kaggle Input (/kaggle/input/**), Colab, and local
data_dir = get_default_data_dir()

# Check if 'train' or 'Train' exists in the detected path
train_exists = (data_dir / "train").exists() or (data_dir / "Train").exists()

if train_exists:
    print(f"[SUCCESS] Dataset detected and ready at: {data_dir}")
    subfolders = [p.name for p in data_dir.iterdir() if p.is_dir()]
    print(f"  Detected subfolders: {subfolders}")
else:
    # 2. Check for zip file in current folder (typical Colab workflow)
    zip_path = "simplicity_dataset.zip"
    target_dir = "Dataset/SIMPLIcity"
    if os.path.exists(zip_path):
        print(f"Extracting {zip_path} to {target_dir}...")
        os.makedirs(target_dir, exist_ok=True)
        with zipfile.ZipFile(zip_path, "r") as zip_ref:
            zip_ref.extractall(target_dir)
        data_dir = Path(target_dir).resolve()
        print(f"[SUCCESS] Extraction complete! Dataset ready at: {data_dir}")
    else:
        print(f"[NOTICE] Dataset 'train/' folder not found at: {data_dir}")
        # If running on Kaggle, list the exact contents of /kaggle/input to inspect paths
        if os.path.exists("/kaggle/input"):
            print("\nInspecting /kaggle/input contents:")
            for root, dirs, _ in os.walk("/kaggle/input", followlinks=True):
                print(f"  Folder: {root}")
                if dirs:
                    print(f"    Subdirectories: {dirs}")
        print("\nTip: You can set the path explicitly if needed:")
        print("     os.environ['DATA_DIR'] = '/kaggle/input/datasets/arfiadi/simplicity'")


## 4. Run Comprehensive Dataset Audit
Verify integrity of images and distribution across the 10 classes in train, validation, and test splits.

In [ ]:
!python scripts/audit_data.py

---

## 5. Experiment 1: Baseline Custom CNN (Slide 6 Assignment)
**Architecture & Setup:**
- 1st Conv: 64 Filters (3x3, pad=1) + ReLU + MaxPool(2x2)
- 2nd Conv: 128 Filters (3x3, pad=1) + ReLU + MaxPool(2x2)
- Linear(131072, 128) + ReLU -> Linear(128, 10)
- Control Group: No heavy photometric jitter, CrossEntropy without label smoothing.

In [ ]:
# ==========================================================================
# CELL KHUSUS: MENAMPILKAN KONFIGURASI LENGKAP EKSPERIMEN 1 SEBELUM TRAINING
# ==========================================================================
from simplicity_cnn.config.schema import print_config_summary

config_path_exp1 = "configs/baseline_custom.yaml"
cfg_exp1 = print_config_summary(config_path_exp1)

In [ ]:
# Eksekusi Training Eksperimen 1 (Baseline Custom)
!python scripts/train.py --config configs/baseline_custom.yaml

### Display Baseline Curves & Confusion Matrix

In [ ]:
from IPython.display import Image, display

print("=== Baseline CNN Gradient Descent Curves ===")
display(Image("outputs/baseline_custom/gradient_descent_summary.png"))

print("=== Baseline CNN Test Confusion Matrix ===")
display(Image("outputs/baseline_custom/evaluation/confusion_matrix.png"))

---

## 6. Experiment 2: Regularized Custom CNN from Scratch (EDA Treatment)
**Architecture & EDA Mitigations:**
- 3 Conv Stages with Batch Normalization & Dropout
- Adaptive Average Pooling (512 features) to prevent FC parameter explosion
- Photometric: ColorJitter (0.25) & RandomErasing (0.20) to counter uniform background shortcuts
- Geometric: RandomResizedCrop (0.75-1.0) without vertical flip
- Training: Label Smoothing (0.10), Weight Decay (5e-4), and Macro F1 Early Stopping

In [ ]:
# ==========================================================================
# CELL KHUSUS: MENAMPILKAN KONFIGURASI LENGKAP EKSPERIMEN 2 SEBELUM TRAINING
# ==========================================================================
from simplicity_cnn.config.schema import print_config_summary

config_path_exp2 = "configs/custom_scratch_regularized.yaml"
cfg_exp2 = print_config_summary(config_path_exp2)

In [ ]:
# Eksekusi Training Eksperimen 2 (Custom Scratch Regularized)
!python scripts/train.py --config configs/custom_scratch_regularized.yaml

### Display Regularized Custom CNN Curves & Confusion Matrix

In [ ]:
print("=== Regularized Custom CNN Gradient Descent Curves ===")
display(Image("outputs/custom_scratch_regularized/gradient_descent_summary.png"))

print("=== Regularized Custom CNN Test Confusion Matrix ===")
display(Image("outputs/custom_scratch_regularized/evaluation/confusion_matrix.png"))

---

## 7. Experiment 3: Transfer Learning Benchmark (ResNet18)
**Architecture & Setup:**
- Backbone: ResNet18 with ImageNet-1K Pretrained Weights
- Fine-tuning: Lower learning rate (3e-4) with Cosine Annealing
- Early Stopping: Monitored on Validation Macro F1

In [ ]:
# ==========================================================================
# CELL KHUSUS: MENAMPILKAN KONFIGURASI LENGKAP EKSPERIMEN 3 SEBELUM TRAINING
# ==========================================================================
from simplicity_cnn.config.schema import print_config_summary

config_path_exp3 = "configs/resnet18_transfer.yaml"
cfg_exp3 = print_config_summary(config_path_exp3)

In [ ]:
# Eksekusi Training Eksperimen 3 (Transfer Learning Benchmark)
!python scripts/train.py --config configs/resnet18_transfer.yaml

### Display ResNet18 Curves & Confusion Matrix

In [ ]:
print("=== ResNet18 Gradient Descent Curves ===")
display(Image("outputs/resnet18_transfer/gradient_descent_summary.png"))

print("=== ResNet18 Test Confusion Matrix ===")
display(Image("outputs/resnet18_transfer/evaluation/confusion_matrix.png"))

---

## 8. Comparative Results Summary
Generate comparative analysis table across all experiments.

In [ ]:
import json
import os

import pandas as pd

models = ["baseline_custom", "custom_scratch_regularized", "resnet18_transfer"]
rows = []

for m in models:
    report_p = f"outputs/{m}/evaluation/classification_report.json"
    hist_p = f"outputs/{m}/history.json"
    if os.path.exists(report_p) and os.path.exists(hist_p):
        with open(report_p) as f:
            rep = json.load(f)
        with open(hist_p) as f:
            hist = json.load(f)
        rows.append(
            {
                "Experiment": m,
                "Best Val Acc (%)": round(max(hist["val_acc"]), 2),
                "Test Accuracy (%)": round(rep["accuracy"] * 100, 2),
                "Macro F1": round(rep["macro avg"]["f1-score"], 4),
                "Macro Precision": round(rep["macro avg"]["precision"], 4),
                "Macro Recall": round(rep["macro avg"]["recall"], 4),
            }
        )

if rows:
    df_summary = pd.DataFrame(rows)
    display(df_summary)
else:
    print("Run experiments above to populate the summary table.")